# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, lower

In [0]:
renamed_table= {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "customer_firstname",
    "cst_lastname": "customer_lastname",
    "cst_marital_status": "customer_marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "customer_create_date"
}

# Read bronze table

In [0]:
df = spark.table("bike_data.bronze.cust_info_raw")
df.display()

# Transformations

## Normalizations

In [0]:
df = (df
      .withColumn("cst_key", lower(col("cst_key")))
      .withColumn("cst_marital_status", lower(col("cst_marital_status")))
      .withColumn("cst_gndr", lower(col("cst_gndr")))
)

## Trimming

In [0]:
for item in df.schema.fields:
    if isinstance(item.dataType, StringType):
        df = df.withColumn(item.name, trim(col(item.name))) 
df.display()

## Updating field names

In [0]:
df = (df
      .withColumn(
          "cst_marital_status", 
                  F.when(col("cst_marital_status") == "s", "single")
                  .when(col("cst_marital_status") == "m", "married")
                  .otherwise(col("cst_marital_status"))
      )
      .withColumn("cst_gndr",
                   F.when(col("cst_gndr") == "m", "male")
                   .when(col("cst_gndr") == "f", "female")
                   .otherwise(col("cst_gndr"))
      )
)

df.display()
      

## Rename columns

In [0]:
for old_name, new_name in renamed_table.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()

# Write to silver table

In [0]:
(
    df.write
    .mode("overwrite")
    .saveAsTable("bike_data.silver.crm_customer_information")
)